# CHSH Lab simulations

This notebook mirrors the revised paper and publication figures for CHSH Lab. The claim is narrow: a postselected CHSH value above 2 is not exclusionary evidence against local models when the selection rule itself is not independently constrained.


## What this notebook covers

1. Reference scales for the classical bound, Wang et al. 2025, the Tsirelson bound, and the local postselection counterexample.
2. Efficiency context for interpreting claims made in an ultra-low-efficiency regime.
3. The exact postselection curve used by the website widget and the revised paper.
4. Correlator inflation under the exact local filter.
5. A small Monte Carlo reproduction of the paper point.

The optics result can be interesting without the Bell inference being decisive. This notebook is about that distinction.


In [ ]:
import math
import random
from statistics import mean, stdev

import matplotlib.pyplot as plt
import numpy as np

BG = '#f7f4ee'
INK = '#1d2430'
MUTED = '#5f6773'
CRIMSON = '#9f3d3d'
AMBER = '#b8842f'
BLUE = '#3f7192'
GREEN = '#537a5a'
SAND = '#dfd5c5'

plt.rcParams.update({
    'figure.facecolor': BG,
    'axes.facecolor': BG,
    'axes.edgecolor': SAND,
    'axes.labelcolor': INK,
    'axes.titlecolor': INK,
    'text.color': INK,
    'xtick.color': MUTED,
    'ytick.color': MUTED,
    'grid.color': '#d8d1c5',
    'font.size': 11,
    'font.family': 'DejaVu Serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

ETA_CRIT = 2 / (1 + math.sqrt(2))
RAW_MAG = 0.5
WANG_S = 2.275
WANG_EFF = 1e-18


def selected_magnitude(p_lo: float) -> float:
    p_hi = 1 - p_lo
    num = ((p_hi + p_lo) * RAW_MAG) + (p_hi - p_lo)
    den = (p_hi + p_lo) + (RAW_MAG * (p_hi - p_lo))
    return num / den


def post_s(p_lo: float) -> float:
    return 4 * selected_magnitude(p_lo)


def accept_rate(p_lo: float) -> float:
    p_hi = 1 - p_lo
    return ((p_hi + p_lo) + (RAW_MAG * (p_hi - p_lo))) / 2

paper_p_lo = 0.10
paper_s = post_s(paper_p_lo)
paper_eta = accept_rate(paper_p_lo)
paper_corr = selected_magnitude(paper_p_lo)

paper_s, paper_eta, paper_corr


## 1. Reference scales

This first figure is the shortest summary of the revised argument. Wang et al. report a super-classical value, but the exact local counterexample used in the site and paper produces a still larger postselected value. That means the number alone cannot settle the foundational question.


In [ ]:
labels = [
    'Classical bound',
    'Wang et al. 2025',
    'Tsirelson bound',
    'Local toy model after selection',
]
values = [2.0, WANG_S, 2 * math.sqrt(2), paper_s]
colors = [AMBER, BLUE, GREEN, CRIMSON]

y = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(8.6, 4.6))
ax.barh(y, values, color=colors, edgecolor=INK, linewidth=0.9)
ax.set_yticks(y, labels)
ax.invert_yaxis()
ax.set_xlim(0, 4.05)
ax.set_xlabel('CHSH value S')
ax.set_title('Reference scales for the revised paper')
ax.grid(axis='x', linestyle='--', linewidth=0.8, alpha=0.8)
ax.axvline(2.0, color=AMBER, linestyle='--', linewidth=1.2)
ax.axvline(2 * math.sqrt(2), color=GREEN, linestyle=':', linewidth=1.3)
for yi, value in enumerate(values):
    ax.text(min(value + 0.06, 3.92), yi, f'{value:.3f}', va='center', ha='left', fontsize=10)
plt.show()


## 2. Efficiency context

The revised paper leans harder on the older detector-loophole literature because Wang et al. operate in an extraordinarily low-efficiency regime. The point is not that low efficiency automatically invalidates every result. The point is that low efficiency raises the evidentiary bar for claims about excluding local models.


In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 4.8))
ax.set_xscale('log')
ax.set_xlim(1e-19, 1.2)
ax.set_ylim(0, 1)
ax.set_yticks([])
ax.set_xlabel('Effective efficiency or threshold')
ax.set_title('Efficiency context for interpreting S > 2')
ax.grid(axis='x', which='both', linestyle='--', linewidth=0.8, alpha=0.7)
ax.axvspan(1e-19, 2 / 3, color=CRIMSON, alpha=0.08)
ax.axvspan(2 / 3, 1.2, color=GREEN, alpha=0.05)
markers = [
    ('Wang et al. eta ~ 1e-18', WANG_EFF, CRIMSON, 0.18),
    ('Eberhard threshold ~ 0.667', 2 / 3, AMBER, 0.40),
    ('Garg-Mermin threshold ~ 0.828', ETA_CRIT, BLUE, 0.62),
    ('Unit efficiency', 1.0, MUTED, 0.84),
]
for label, value, color, y in markers:
    ax.axvline(value, color=color, linewidth=2)
    ax.scatter([value], [y], s=42, color=color, zorder=3)
    ax.text(value * 1.06, y, label, color=color, va='center', fontsize=9)
plt.show()


## 3. Exact postselection curve used by the website and paper

The website used to show a proxy. It now uses the exact toy-model formulas below. The revised paper and this notebook use the same definitions.

\[
E^{\mathrm{sel}}(p_{lo}) = 
rac{	frac{3}{2} - 2p_{lo}}{	frac{3}{2} - p_{lo}}, \qquad
S^{\mathrm{sel}}(p_{lo}) = 4 E^{\mathrm{sel}}(p_{lo}), \qquad
\eta(p_{lo}) = 	frac{3}{4} - 	frac{1}{2} p_{lo}.
\]


In [ ]:
p_lo = np.linspace(0.0, 0.5, 400)
s_sel = np.array([post_s(v) for v in p_lo])
eta = np.array([accept_rate(v) for v in p_lo])

fig, ax1 = plt.subplots(figsize=(8.6, 5.0))
ax2 = ax1.twinx()
ax1.plot(p_lo, s_sel, color=CRIMSON, linewidth=2.4, label='Selected S')
ax2.plot(p_lo, eta, color=BLUE, linewidth=2.0, linestyle='--', label='Acceptance')
ax1.axhline(2.0, color=AMBER, linestyle='--', linewidth=1.2)
ax1.axhline(2 * math.sqrt(2), color=GREEN, linestyle=':', linewidth=1.2)
ax2.axhline(ETA_CRIT, color=AMBER, linestyle='-.', linewidth=1.0)
ax1.scatter([paper_p_lo], [paper_s], color=CRIMSON, s=48, zorder=4)
ax2.scatter([paper_p_lo], [paper_eta], color=BLUE, s=40, zorder=4)
ax1.set_xlim(0, 0.5)
ax1.set_ylim(1.95, 4.05)
ax2.set_ylim(0.5, 0.78)
ax1.set_xlabel('Disfavored-sign keep rate p_lo')
ax1.set_ylabel('Selected CHSH value S')
ax2.set_ylabel('Acceptance rate')
ax1.set_title('Exact local counterexample used in the website and notebook')
ax1.grid(axis='both', linestyle='--', linewidth=0.8, alpha=0.7)
plt.show()

print({'paper_p_lo': paper_p_lo, 'paper_s': round(paper_s, 6), 'paper_eta': round(paper_eta, 6)})


## 4. Correlator inflation

The local source starts with raw correlators of magnitude 1/2, so the raw CHSH value is exactly 2. The selected correlators become 13/14 at the paper point. The selection step is therefore where the apparent excess is generated.


In [ ]:
settings = ['E00', 'E01', 'E10', 'E11']
raw = np.array([0.5, 0.5, 0.5, -0.5])
selected = np.array([paper_corr, paper_corr, paper_corr, -paper_corr])
x = np.arange(len(settings))
width = 0.34

fig, ax = plt.subplots(figsize=(8.6, 4.8))
ax.bar(x - width / 2, raw, width=width, color=BLUE, edgecolor=INK, label='Raw local correlators')
ax.bar(x + width / 2, selected, width=width, color=CRIMSON, edgecolor=INK, label='Selected correlators')
ax.axhline(0, color=SAND, linewidth=1)
ax.set_xticks(x, settings)
ax.set_ylim(-1.05, 1.05)
ax.set_ylabel('Correlation value')
ax.set_title('Selection inflates the correlators, not the locality class')
ax.grid(axis='y', linestyle='--', linewidth=0.8, alpha=0.7)
ax.legend(frameon=False)
plt.show()


## 5. Small Monte Carlo reproduction

This simulation is not trying to reconstruct Wang et al.'s apparatus. It reproduces the exact local toy filter many times and checks that the sample mean settles near the analytic paper point.


In [ ]:
raw_patterns = [
    (1, 1, 1, -1),
    (1, 1, -1, 1),
    (1, -1, 1, 1),
    (-1, 1, 1, 1),
]


def one_run(trials=20000, p_lo=paper_p_lo, seed=0):
    rng = random.Random(seed)
    kept = [[], [], [], []]
    accepted = 0
    for _ in range(trials):
        a0, a1, b0, b1 = raw_patterns[rng.randrange(len(raw_patterns))]
        products = [a0*b0, a0*b1, a1*b0, a1*b1]
        for idx, prod in enumerate(products):
            favored = (idx < 3 and prod == 1) or (idx == 3 and prod == -1)
            keep_prob = 1 - p_lo if favored else p_lo
            if rng.random() < keep_prob:
                kept[idx].append(prod)
                accepted += 1
    est = [sum(vals) / len(vals) for vals in kept]
    s = abs(est[0] + est[1] + est[2] - est[3])
    eta = accepted / (trials * 4)
    return s, eta

runs = [one_run(seed=i) for i in range(20)]
s_values = [r[0] for r in runs]
eta_values = [r[1] for r in runs]
summary = {
    'mean_S': mean(s_values),
    'std_S': stdev(s_values),
    'mean_eta': mean(eta_values),
    'std_eta': stdev(eta_values),
}
summary


## Takeaway

The revised paper does not claim that Wang et al. observed nothing. It claims something narrower and more defensible: in an ultra-low-efficiency postselected regime, the Bell-style scalar is not self-interpreting. The publication figures, arXiv draft, and website widget now all tell that same story.
